# Lakelink Fiber Network Scenario Twin: corridor forecast (ML notebook 2 of 3, D-009)
Owner: @ml-engineer. Tag: `telco_project: network_scenario_twin`. Spec: `resources/model_design.md` sections 1-2.

* **Section 1 (P3-FEATURES):** builds `cdm_tmforum.network_scenario_twin.corridor_feature_daily` from gold `corridor_utilization_daily` + active `service`. Grain: corridor x day, PK (corridor_id, feature_date).
* **Section 2 (P3-FORECAST-MODEL, done 2026-11-23):** log-growth champion `log_growth_v1`, LightGBM challenger `lgbm_global_v1`, linear baseline; writes `sim_capacity_forecast` (480 rows).

In [0]:
%sql
-- P3-FEATURES: corridor feature table with the AI growth regressor (ln1p_t) and exogenous ai_growth_index
CREATE OR REPLACE TABLE cdm_tmforum.network_scenario_twin.corridor_feature_daily
COMMENT 'Lakelink Fiber Network Scenario Twin: corridor-level ML feature table (P3-FEATURES, @ml-engineer). Grain: 1 row per corridor x UTC day. Source: gold corridor_utilization_daily + service (active only). Includes the AI growth regressor ln1p_t and exogenous ai_growth_index. Feeds the corridor utilization forecast (sim_capacity_forecast). Simulation output, not an inventory record.'
AS
WITH base AS (SELECT * FROM cdm_tmforum.network_scenario_twin.corridor_utilization_daily),
ai AS (
  SELECT corridor_id, ep_dc_mw_next12 / NULLIF(ep_dc_mw_online_now, 0) AS ai_ratio
  FROM base GROUP BY corridor_id, ep_dc_mw_next12, ep_dc_mw_online_now),
ai_norm AS (
  SELECT corridor_id,
         COALESCE((ai_ratio - MIN(ai_ratio) OVER ()) / NULLIF(MAX(ai_ratio) OVER () - MIN(ai_ratio) OVER (), 0), 0.0) AS ai_growth_index
  FROM ai),
svc AS (
  SELECT corridor_id, COUNT(*) AS n_services, CAST(SUM(mrr) AS DOUBLE) AS mrr_on_corridor
  FROM cdm_tmforum.network_scenario_twin.service WHERE status = 'active' GROUP BY corridor_id)
SELECT b.corridor_id, b.util_date AS feature_date,
  b.util_mean_pct, b.util_p95_pct, b.util_max_pct, b.util_7d_mean_pct, b.util_28d_mean_pct,
  regr_slope(b.util_mean_pct, b.t_days) OVER (PARTITION BY b.corridor_id ORDER BY b.util_date ROWS BETWEEN 6 PRECEDING AND CURRENT ROW) AS slope_7d,
  regr_slope(b.util_mean_pct, b.t_days) OVER (PARTITION BY b.corridor_id ORDER BY b.util_date ROWS BETWEEN 27 PRECEDING AND CURRENT ROW) AS slope_28d,
  b.t_days, b.ln1p_t,
  a.ai_growth_index, b.ai_signal_missing,
  b.capacity_gbps, b.route_type, b.is_diverse_route,
  b.n_spans, COALESCE(s.n_services, 0) AS n_services, COALESCE(s.mrr_on_corridor, 0.0) AS mrr_on_corridor,
  b.latency_mean_ms, b.optical_margin_min_db,
  b.origin, b.simulation_run_id
FROM base b
LEFT JOIN ai_norm a USING (corridor_id)
LEFT JOIN svc s USING (corridor_id)

In [0]:
T = 'cdm_tmforum.network_scenario_twin.corridor_feature_daily'
spark.sql(f"ALTER TABLE {T} ALTER COLUMN corridor_id SET NOT NULL")
spark.sql(f"ALTER TABLE {T} ALTER COLUMN feature_date SET NOT NULL")
spark.sql(f"ALTER TABLE {T} ADD CONSTRAINT corridor_feature_daily_pk PRIMARY KEY (corridor_id, feature_date)")
comments = {
 'corridor_id':'Corridor key (FK corridor.corridor_id), e.g. DEN-SLC. Part of PK.',
 'feature_date':'UTC day of the feature row (2026-08-18 onward). Part of PK.',
 'util_mean_pct':'Daily mean corridor utilization %, capacity-weighted across member spans.',
 'util_p95_pct':'Daily p95 corridor utilization %.',
 'util_max_pct':'Daily max corridor utilization %.',
 'util_7d_mean_pct':'Trailing 7-day mean utilization %; forecast target.',
 'util_28d_mean_pct':'Trailing 28-day mean utilization %.',
 'slope_7d':'OLS slope of util_mean_pct vs t_days over trailing 7 days (pct-points/day); NULL on first day.',
 'slope_28d':'OLS slope of util_mean_pct vs t_days over trailing 28 days (pct-points/day); NULL on first day.',
 't_days':'Days since 2026-08-18.',
 'ln1p_t':'AI growth regressor ln(1 + k_ref * t_days), k_ref = 0.108783.',
 'ai_growth_index':'Exogenous AI growth signal: ep_dc_mw_next12 / ep_dc_mw_online_now from fiber_capital_intelligence, min-max normalized to [0,1] across corridors; 0 when missing.',
 'ai_signal_missing':'True when the corridor has no fiber_capital AI growth signal (ai_growth_index set to 0).',
 'capacity_gbps':'Corridor capacity in Gbps.',
 'route_type':'long_haul or metro.',
 'is_diverse_route':'True if the corridor is sold as a diverse route.',
 'n_spans':'Number of member fiber spans.',
 'n_services':'Count of active services on the corridor.',
 'mrr_on_corridor':'Sum of monthly recurring revenue (USD) of active services on the corridor.',
 'latency_mean_ms':'Daily mean latency (ms).',
 'optical_margin_min_db':'Daily minimum optical margin (dB).',
 'origin':'Provenance: SYNTHETIC or MIXED.',
 'simulation_run_id':'Simulation run id, e.g. nst-sim-001.'}
for c, v in comments.items():
    spark.sql(f"ALTER TABLE {T} ALTER COLUMN {c} COMMENT '{v.replace(chr(39), chr(39)*2)}'")
spark.sql(f"ALTER TABLE {T} SET TAGS ('domain'='operations','telco_sensitivity'='internal')")
# telco_project=network_scenario_twin is currently rejected by the governed tag policy; apply it once the value is allowed:
try:
    spark.sql(f"ALTER TABLE {T} SET TAGS ('telco_project'='network_scenario_twin')")
except Exception as e:
    print('telco_project tag not applied (tag policy):', str(e)[:160])

In [0]:
%sql
-- Expect: n = gold rows (3,600 = 40 x 90), pk = n, null_tgt = 0, ai index in [0,1], 0 bad_missing
SELECT count(*) n, count(DISTINCT corridor_id) corridors, count(DISTINCT corridor_id, feature_date) pk,
       min(feature_date) dmin, max(feature_date) dmax,
       sum(CASE WHEN util_7d_mean_pct IS NULL THEN 1 ELSE 0 END) null_tgt,
       min(ai_growth_index) ai_min, max(ai_growth_index) ai_max,
       sum(CASE WHEN ai_signal_missing AND ai_growth_index <> 0 THEN 1 ELSE 0 END) bad_missing
FROM cdm_tmforum.network_scenario_twin.corridor_feature_daily

In [0]:
# Section 2 (P3-FORECAST-MODEL), restored 2026-11-23 by @ml-engineer (P3-FORECAST-SEC2-RESTORE / P3-FORECAST-CODE-PERSIST).
# Lakelink Fiber Network Scenario Twin. Forecast/simulation output, not an inventory record. Tag: telco_project: network_scenario_twin.
# Reproduces champion run ab906ec88298465aa475e168acefd615 (curve_params.json) from corridor_feature_daily v25:
#   * target = util_7d_mean_pct, centered: t = t_days - 3 (lag_center_days=3); first 6 days dropped (warmup_days=6) -> t in [3, 86], t_max = 86
#   * u(t) = a + b*ln(1 + k*t), bounded NLS a in [0,100], b in [0,200], k in [1e-4, 5]
#   * fallback: b*ln(1 + k*t_max) < 2 -> damped_linear, damped_slope = 0.5 * OLS slope (centered series vs t)
#   * AI lever: growth above u(t0) scaled by (1 + 0.25*ai_growth_index) from 2027-06-01 (t0 = 287)
#   * band: split-conformal, sqrt-horizon. Calibration: fit t_days <= 41, score t_days 42..62, q80 of |r|/sqrt(h)
#   * holdout: fit t_days <= 62, test t_days 63..89 (1,080 points) -> MAPE, per-route_type MAPE, P10-P90 coverage
#   * exhaustion: first day P50 >= 95% within 365 days of 2026-11-16; P10 date from upper band, P90 date from lower band
# LightGBM challenger lgbm_global_v1 is NOT retrained here: its artifact is preserved in MLflow (models:/m-e6fe81b408a34e98ad6e191dd2c8a5ea,
# UC corridor_util_forecast v2 @challenger, holdout MAPE 0.481, never promoted).
import json, datetime as dt, numpy as np, pandas as pd, mlflow
from scipy.optimize import curve_fit

WRITE = False          # True = overwrite sim_capacity_forecast (480 rows). False = dry run, compare with the published table
LOG_MLFLOW = False     # True = log a new run to experiments/nst_models
FEATURE_VERSION = 25   # corridor_feature_daily version used by the champion run (param feature_table_version)
S = "cdm_tmforum.network_scenario_twin"
CHAMPION_RUN = "ab906ec88298465aa475e168acefd615"
T0, AS_OF = dt.date(2026, 8, 18), dt.date(2026, 11, 16)
CFG = dict(lag_center_days=3, warmup_days=6, ai_growth_weight=0.25, lever_start_t=(dt.date(2027, 6, 1) - T0).days,
           exhaustion_threshold_pct=95.0, fallback_min_rise=2.0, damp=0.5, calib_fit_max_t=41, holdout_train_max_t=62, q=0.8)

fd = (spark.read.option("versionAsOf", FEATURE_VERSION).table(f"{S}.corridor_feature_daily")
      .select("corridor_id", "t_days", "util_7d_mean_pct", "route_type", "ai_growth_index").toPandas())
fd = fd[fd.t_days >= CFG["warmup_days"]].copy()
fd["t"] = fd.t_days.astype(float) - CFG["lag_center_days"]
fd = fd.sort_values(["corridor_id", "t"])

logc = lambda t, a, b, k: a + b * np.log1p(k * t)
def fit_corridor(g):
    t, y = g.t.values, g.util_7d_mean_pct.values.astype(float)
    (a, b, k), _ = curve_fit(logc, t, y, p0=[y[0], 10, 0.05], bounds=([0, 0, 1e-4], [100, 200, 5]), maxfev=20000)
    tm = float(t.max()); damped = b * np.log1p(k * tm) < CFG["fallback_min_rise"]
    return {"mode": "damped_linear" if damped else "log_growth", "a": float(a), "b": float(b), "k": float(k),
            "damped_slope": float(CFG["damp"] * np.polyfit(t, y, 1)[0]) if damped else None,
            "ai_growth_index": float(g.ai_growth_index.iloc[0]), "t_max": tm}
def u_base(p, t):
    t = np.asarray(t, float)
    if p["mode"] == "log_growth": return logc(t, p["a"], p["b"], p["k"])
    tm = p["t_max"]; return logc(tm, p["a"], p["b"], p["k"]) + p["damped_slope"] * (t - tm)
def u50(p, t, lever=True):
    t = np.asarray(t, float); ub = u_base(p, t)
    if not lever: return ub
    t0 = CFG["lever_start_t"]; u0 = u_base(p, t0)
    return np.where(t >= t0, u0 + (ub - u0) * (1 + CFG["ai_growth_weight"] * p["ai_growth_index"]), ub)
def split_eval(train_max, test_lo, test_hi):
    out = []
    for cid, g in fd.groupby("corridor_id"):
        tr, te = g[g.t_days <= train_max], g[(g.t_days >= test_lo) & (g.t_days <= test_hi)]
        p = fit_corridor(tr)
        out.append(pd.DataFrame({"corridor_id": cid, "route_type": g.route_type.iloc[0], "h": te.t.values - p["t_max"],
                                 "y": te.util_7d_mean_pct.values, "yhat": u50(p, te.t.values, lever=False)}))
    return pd.concat(out, ignore_index=True)

# 1. Conformal band width
cal = split_eval(CFG["calib_fit_max_t"], CFG["calib_fit_max_t"] + 1, CFG["holdout_train_max_t"])
q80 = float(np.quantile(np.abs(cal.y - cal.yhat) / np.sqrt(cal.h), CFG["q"]))
# 2. Holdout accuracy and calibration
ho = split_eval(CFG["holdout_train_max_t"], CFG["holdout_train_max_t"] + 1, 89)
ho["ape"] = (ho.y - ho.yhat).abs() / ho.y * 100
metrics = {"holdout_mape_pct": ho.ape.mean(), **{f"holdout_mape_{k}_pct": v for k, v in ho.groupby("route_type").ape.mean().items()},
           "band_coverage_p10_p90": float((np.abs(ho.y - ho.yhat) <= q80 * np.sqrt(ho.h)).mean()), "band_q80": q80}
# 3. Full fit -> curve params
curve_params = {cid: fit_corridor(g) for cid, g in fd.groupby("corridor_id")}

# 4. 12-month forecast rows (2026-12 .. 2027-11) + exhaustion dates
qr = round(q80, 5); thr = CFG["exhaustion_threshold_pct"]
te = np.arange((AS_OF - T0).days + 1, (AS_OF - T0).days + 366)
first = lambda x: T0 + dt.timedelta(int(te[np.argmax(x >= thr)])) if (x >= thr).any() else None
months = [dt.date(2026 + (11 + i) // 12, (11 + i) % 12 + 1, 1) for i in range(12)]
now = dt.datetime.now(dt.timezone.utc).replace(tzinfo=None); rows = []
for cid, p in curve_params.items():
    ue = u50(p, te); we = qr * np.sqrt(np.maximum(te - p["t_max"], 0)); e50 = first(ue)
    for m in months:
        nxt = dt.date(m.year + (m.month == 12), m.month % 12 + 1, 1)
        ts = np.arange((m - T0).days, (nxt - T0).days); u = u50(p, ts); w = qr * np.sqrt(np.maximum(ts - p["t_max"], 0))
        rows.append({"forecast_id": f"FC-{cid}-{m:%Y%m}", "corridor_id": cid, "month": m,
                     "forecast_utilization_pct": round(float(u.mean()), 3), "forecast_p10_pct": round(float((u - w).mean()), 3),
                     "forecast_p90_pct": round(float((u + w).mean()), 3), "exhaustion_date": e50,
                     "exhaustion_date_p10": first(ue + we), "exhaustion_date_p90": first(ue - we),
                     "days_to_exhaustion": None if e50 is None else (e50 - AS_OF).days, "exhaustion_threshold_pct": thr,
                     "model_name": "corridor_util_forecast", "model_version": "log_growth_v1", "model_alias": "champion",
                     "mlflow_run_id": CHAMPION_RUN, "forecast_generated_at": now, "origin": "SYNTHETIC", "simulation_run_id": "nst-sim-001"})
fc = pd.DataFrame(rows)

# 5. Linear baseline (logged, never promoted): OLS over the window, no curvature
bl_ho = []
for cid, g in fd.groupby("corridor_id"):
    tr, tst = g[g.t_days <= CFG["holdout_train_max_t"]], g[g.t_days > CFG["holdout_train_max_t"]]
    s, i = np.polyfit(tr.t, tr.util_7d_mean_pct, 1); bl_ho.append((tst.util_7d_mean_pct - (i + s * tst.t)).abs() / tst.util_7d_mean_pct * 100)
g = fd[fd.corridor_id == "DEN-SLC"]; s, i = np.polyfit(g.t, g.util_7d_mean_pct, 1)
bl_den = T0 + dt.timedelta(int(te[np.argmax(i + s * te >= thr)])) if (i + s * te >= thr).any() else None
baseline = {"holdout_mape_pct": float(pd.concat(bl_ho).mean()), "den_slc_exhaustion_date": str(bl_den),
            "den_slc_days_to_exhaustion": None if bl_den is None else (bl_den - AS_OF).days}

# 6. Gates (model_design.md s2.7) + reproduction of the champion run
den = fc[fc.corridor_id == "DEN-SLC"].iloc[0]
false_exh = int(fc[(fc.corridor_id != "DEN-SLC") & fc.exhaustion_date.notna()]
                .query("exhaustion_date < @dt.date(2027, 5, 16)").corridor_id.nunique())
pub_cp = json.load(open(mlflow.artifacts.download_artifacts(run_id=CHAMPION_RUN, artifact_path="curve_params.json", dst_path="/tmp/nst_fc_cp")))
max_db = max(abs(curve_params[c]["b"] - pub_cp[c]["b"]) for c in pub_cp)
modes_match = all(curve_params[c]["mode"] == pub_cp[c]["mode"] for c in pub_cp)
pub = spark.table(f"{S}.sim_capacity_forecast").toPandas().sort_values("forecast_id").reset_index(drop=True)
new = fc.sort_values("forecast_id").reset_index(drop=True)
num = ["forecast_utilization_pct", "forecast_p10_pct", "forecast_p90_pct"]
dcols = ["exhaustion_date", "exhaustion_date_p10", "exhaustion_date_p90"]
same_ids = list(pub.forecast_id) == list(new.forecast_id)
max_err = float(np.abs(pub[num].values.astype(float) - new[num].values.astype(float)).max()) if same_ids else None
dates_match = same_ids and all((pd.isna(a) and b is None) or str(a) == str(b) for c in dcols for a, b in zip(pub[c], new[c]))
gates = {"gate_acceptance_q2": dt.date(2027, 3, 1) <= den.exhaustion_date <= dt.date(2027, 3, 31),
         "gate_false_exhaustion_0": false_exh == 0,
         "gate_holdout_mape": metrics["holdout_mape_pct"] <= 3 and all(v <= 5 for k, v in metrics.items() if k.startswith("holdout_mape_") and k != "holdout_mape_pct"),
         "gate_row_count_480": bool(len(fc) == 480 and fc.corridor_id.nunique() == 40 and fc.groupby("corridor_id").size().eq(12).all()),
         "repro_curve_params_db_lt_0_01": max_db < 0.01 and modes_match,
         "repro_q80_0_08035": round(q80, 5) == 0.08035,
         "repro_exhaustion_dates_match": bool(dates_match),
         "repro_published_table": bool(same_ids and max_err <= 0.001 + 1e-9 and dates_match)}  # 1e-9: 3-dp rounding ties
# band coverage 0.912 sits above the 0.70-0.90 spec gate (known; tracked in P3-SLICE-TESTING), so it is reported, not gated
print(json.dumps({"metrics": {k: round(v, 6) for k, v in metrics.items()}, "baseline": baseline,
                  "den_slc": {"exhaustion_date": str(den.exhaustion_date), "days": int(den.days_to_exhaustion),
                              "p10": str(den.exhaustion_date_p10), "p90": str(den.exhaustion_date_p90)},
                  "modes": pd.Series([p["mode"] for p in curve_params.values()]).value_counts().to_dict(),
                  "max_abs_db": max_db, "max_abs_forecast_err": max_err, "gates": gates}, indent=1, default=str))
assert all(gates.values()), f"Gate failure: {[k for k, v in gates.items() if not v]}"

if LOG_MLFLOW:
    mlflow.set_experiment("/Users/stephen.hage@databricks.com/network-scenario-twin/experiments/nst_models")
    with mlflow.start_run(run_name="corridor_forecast_log_growth_v1_rerun") as r:
        mlflow.set_tags({"telco_project": "network_scenario_twin", "task": "P3-FORECAST-SEC2-RESTORE", "model_name": "log_growth_v1",
                         "simulation_run_id": "nst-sim-001", "company": "Lakelink Fiber", "reproduces_run": CHAMPION_RUN})
        mlflow.log_params({**CFG, "feature_table_version": FEATURE_VERSION})
        mlflow.log_metrics({**metrics, **{k: float(v) for k, v in gates.items()}, "max_abs_db": max_db, "row_count": len(fc)})
        mlflow.log_dict(curve_params, "curve_params.json")

if WRITE:
    # keep CHAMPION_RUN as mlflow_run_id unless the champion is re-registered; MERGE-free full refresh of a 480-row table
    sdf = spark.createDataFrame(fc).selectExpr(*[f"CAST({c} AS {t}) AS {c}" for c, t in spark.table(f"{S}.sim_capacity_forecast").dtypes])
    sdf.write.mode("overwrite").option("overwriteSchema", "false").saveAsTable(f"{S}.sim_capacity_forecast")
    print("sim_capacity_forecast rows:", spark.table(f"{S}.sim_capacity_forecast").count())

In [0]:
# P3-MODEL-REGISTRY (@ml-engineer 2026-11-23), model_design.md s6. Consolidated code of the session that registered:
#   cdm_tmforum.network_scenario_twin.corridor_util_forecast  v1 @champion log_growth_v1 (pyfunc) | v2 @challenger lgbm_global_v1 (lightgbm)
#   cdm_tmforum.network_scenario_twin.blast_radius_sim        v1 @champion srlg_graph_v1 (pyfunc) | v2 @challenger path_only_v1 (pyfunc, never promoted)
# Both pyfuncs reproduce the published tables exactly (480 forecast rows to 3dp + all exhaustion dates; 500 sim rows, every column).
# Requires: %pip install lightgbm. Lakelink Fiber; simulation/forecast outputs, not inventory records.
import json, os, datetime as dt, numpy as np, pandas as pd, mlflow, mlflow.pyfunc
from decimal import Decimal, ROUND_HALF_UP
from mlflow.models import infer_signature
mlflow.set_registry_uri("databricks-uc")
S = "cdm_tmforum.network_scenario_twin"; FC = f"{S}.corridor_util_forecast"; BR = f"{S}.blast_radius_sim"
mlflow.set_experiment("/Users/stephen.hage@databricks.com/network-scenario-twin/experiments/nst_models")
FC_RUN, LGBM_MODEL, BR_RUN = "ab906ec88298465aa475e168acefd615", "models:/m-e6fe81b408a34e98ad6e191dd2c8a5ea", "3b9a58ecf03a4bc7be8c3cad4dd25070"
D = "/tmp/nst_reg"; os.makedirs(D, exist_ok=True)

class CorridorForecastModel(mlflow.pyfunc.PythonModel):
    """log_growth_v1: monthly P50 = mean of daily u(t), t = days since 2026-08-18; AI lever on growth from 2027-06-01;
    band = +/- q80*sqrt(t - t_max); exhaustion = first day >= 95% within 365 d of 2026-11-16 (P10 date from upper band)."""
    T0, AS_OF = "2026-08-18", "2026-11-16"
    def load_context(self, context):
        self.cp = json.load(open(context.artifacts["curve_params"])); self.cfg = json.load(open(context.artifacts["model_config"]))
    def _u50(self, p, t):
        def base(tt):
            tt = np.asarray(tt, float)
            if p["mode"] == "log_growth": return p["a"] + p["b"] * np.log1p(p["k"] * tt)
            tm = p["t_max"]; return p["a"] + p["b"] * np.log1p(p["k"] * tm) + p["damped_slope"] * (tt - tm)
        t = np.asarray(t, float); ub = base(t); t0 = self.cfg["lever_start_t"]; u0 = base(t0)
        return np.where(t >= t0, u0 + (ub - u0) * (1 + self.cfg["ai_growth_weight"] * p["ai_growth_index"]), ub)
    def predict(self, context, model_input, params=None):
        T0, AS_OF = dt.date.fromisoformat(self.T0), dt.date.fromisoformat(self.AS_OF)
        q, thr = self.cfg["band_q80"], self.cfg["exhaustion_threshold_pct"]; rows = []
        for cid, m in zip(model_input["corridor_id"], model_input["month"]):
            p = self.cp.get(str(cid)); m = pd.Timestamp(m).date().replace(day=1)
            if p is None:
                rows.append({"corridor_id": cid, "month": str(m), **{k: None for k in ["forecast_utilization_pct", "forecast_p10_pct", "forecast_p90_pct", "exhaustion_date", "exhaustion_date_p10", "exhaustion_date_p90", "days_to_exhaustion"]}}); continue
            nxt = dt.date(m.year + (m.month == 12), m.month % 12 + 1, 1)
            ts = np.arange((m - T0).days, (nxt - T0).days); u = self._u50(p, ts); w = q * np.sqrt(np.maximum(ts - p["t_max"], 0))
            te = np.arange((AS_OF - T0).days + 1, (AS_OF - T0).days + 366); ue = self._u50(p, te); we = q * np.sqrt(np.maximum(te - p["t_max"], 0))
            first = lambda x: str(T0 + dt.timedelta(int(te[np.argmax(x >= thr)]))) if (x >= thr).any() else None
            e50 = first(ue)
            rows.append({"corridor_id": cid, "month": str(m), "forecast_utilization_pct": round(float(u.mean()), 3),
                         "forecast_p10_pct": round(float((u - w).mean()), 3), "forecast_p90_pct": round(float((u + w).mean()), 3),
                         "exhaustion_date": e50, "exhaustion_date_p10": first(ue + we), "exhaustion_date_p90": first(ue - we),
                         "days_to_exhaustion": None if e50 is None else float((dt.date.fromisoformat(e50) - AS_OF).days)})
        return pd.DataFrame(rows)

class BlastRadiusModel(mlflow.pyfunc.PythonModel):
    """srlg_graph_v1 (srlg_aware=True) / path_only_v1 (False); same semantics as src/ml/nst_blast_radius simulate()."""
    def __init__(self, srlg_aware=True): self.srlg_aware = srlg_aware
    def load_context(self, context):
        g = json.load(open(context.artifacts["graph_snapshot"]))
        self.spans = g["spans"]; self.srlg_of = {s["id"]: s["srlg"] for s in g["spans"]}; self.members = {}
        for s in g["spans"]:
            if s["srlg"]: self.members.setdefault(s["srlg"], set()).add(s["id"])
        self.node_parent = {n["id"]: n["parent"] for n in g["nodes"]}; self.sites = set(g["sites"]); self.services = g["services"]
    def _sim(self, rtype, rid):
        if rtype == "span": fs = {rid} if rid in self.srlg_of else set()
        else:
            kids = {rid} if rtype == "node" else {n for n, p in self.node_parent.items() if p == rid}
            fs = {s["id"] for s in self.spans if s["a"] in kids or s["z"] in kids}
        if self.srlg_aware:
            for s in list(fs):
                if self.srlg_of.get(s): fs |= self.members[self.srlg_of[s]]
        Q = Decimal("0.01"); exp = degm = Decimal(0); down, deg, dual = [], 0, set()
        for v in self.services:
            wh, ph, unp = v["w"] in fs, bool(v["p"]) and v["p"] in fs, v["scheme"] == "unprotected"
            if (unp and wh) or (not unp and wh and ph):
                down.append(v); exp += Decimal(v["mrr"]) * Decimal(v["credit"]) / 100
                if v["scheme"] == "dual_path": dual.add(v["cid"])
            elif not unp and wh != ph: deg += 1; degm += Decimal(v["mrr"])
        return {"resource_type": rtype, "resource_id": rid, "known_resource": bool(fs) or (rtype == "site" and rid in self.sites),
                "srlg_expanded_resource_ids": json.dumps(sorted(fs)), "affected_services": len(down), "affected_customers": len({v["cid"] for v in down}),
                "dual_path_both_failed_customers": len(dual), "dual_path_both_failed_customer_ids": json.dumps(sorted(dual)), "degraded_services": deg,
                "sla_exposure_mrr": float(exp.quantize(Q, ROUND_HALF_UP)), "degraded_mrr": float(degm.quantize(Q, ROUND_HALF_UP))}
    def predict(self, context, model_input, params=None):
        return pd.DataFrame([self._sim(str(t).lower(), str(r)) for t, r in zip(model_input["resource_type"], model_input["resource_id"])])

# Artifacts: curve params from the champion forecast run; graph snapshot of the published twin tables (active services only)
cp_path = mlflow.artifacts.download_artifacts(run_id=FC_RUN, artifact_path="curve_params.json", dst_path=D)
json.dump({"band_q80": 0.08035, "exhaustion_threshold_pct": 95.0, "ai_growth_weight": 0.25, "lever_start_t": (dt.date(2027, 6, 1) - dt.date(2026, 8, 18)).days,
           "band_method": "split_conformal_sqrt_h", "source_run_id": FC_RUN}, open(f"{D}/model_config.json", "w"))
pr = spark.table(f"{S}.physical_resource").toPandas(); lr = spark.table(f"{S}.logical_resource").select("resource_id", "span_id").toPandas()
sv = spark.table(f"{S}.service").toPandas(); sla = spark.table(f"{S}.service_level_agreement").select("service_id", "credit_pct").toPandas()
lr_span = dict(zip(lr.resource_id, lr.span_id)); act = sv[sv.status == "active"].merge(sla, on="service_id", how="left")
act["w"] = act.primary_logical_resource_id.map(lr_span); act["p"] = act.protect_logical_resource_id.map(lambda x: lr_span.get(int(x)) if pd.notna(x) else None)
nn = lambda x: None if pd.isna(x) else x
json.dump({"spans": [{"id": r.resource_id, "srlg": nn(r.srlg), "a": nn(r.site_a_id), "z": nn(r.site_z_id)} for r in pr[pr.type == "span"].itertuples()],
           "nodes": [{"id": r.resource_id, "parent": nn(r.parent_site_id)} for r in pr[pr.type == "node"].itertuples()],
           "sites": list(pr[pr.type == "site"].resource_id),
           "services": [{"sid": int(r.service_id), "cid": int(r.customer_id), "scheme": r.protection_scheme, "w": nn(r.w), "p": nn(r.p), "mrr": str(r.mrr),
                         "credit": str(0 if pd.isna(r.credit_pct) else r.credit_pct)} for r in act.itertuples()]}, open(f"{D}/graph_snapshot.json", "w"))

tags = {"telco_project": "network_scenario_twin", "task": "P3-MODEL-REGISTRY", "simulation_run_id": "nst-sim-001", "company": "Lakelink Fiber"}
pip = [f"mlflow=={mlflow.__version__}", "pandas", "numpy"]
fc_in = pd.DataFrame({"corridor_id": ["DEN-SLC", "DAL-PHX"], "month": ["2027-03-01", "2027-03-01"]})
br_in = pd.DataFrame({"resource_type": ["span", "node"], "resource_id": ["S-4471", "N-1001"]})

def log_pyfunc(model, artifacts, example, name, run_name, extra_tags):
    class _C: pass
    ctx = _C(); ctx.artifacts = artifacts; model.load_context(ctx)
    with mlflow.start_run(run_name=run_name) as r:
        mlflow.set_tags({**tags, **extra_tags})
        info = mlflow.pyfunc.log_model(name="model", python_model=model, artifacts=artifacts, signature=infer_signature(example, model.predict(None, example)),
                                       input_example=example, pip_requirements=pip, registered_model_name=name)
    return info.registered_model_version

REGISTER = False   # set True to register new versions; aliases below are then re-pointed
if REGISTER:
    v_fc = log_pyfunc(CorridorForecastModel(), {"curve_params": cp_path, "model_config": f"{D}/model_config.json"}, fc_in, FC, "register_corridor_util_forecast_log_growth_v1", {"role": "champion", "source_run_id": FC_RUN})
    lgbm = mlflow.lightgbm.load_model(LGBM_MODEL); X = pd.DataFrame(np.zeros((2, 6)), columns=[f"Column_{i}" for i in range(6)])
    with mlflow.start_run(run_name="register_corridor_util_forecast_lgbm_global_v1"):
        mlflow.set_tags({**tags, "role": "challenger", "source_logged_model": LGBM_MODEL})
        v_lg = mlflow.lightgbm.log_model(lgbm, name="model", signature=infer_signature(X, lgbm.predict(X)), input_example=X, registered_model_name=FC).registered_model_version
    v_br = log_pyfunc(BlastRadiusModel(True), {"graph_snapshot": f"{D}/graph_snapshot.json"}, br_in, BR, "register_blast_radius_sim_srlg_graph_v1", {"role": "champion", "source_run_id": BR_RUN})
    v_po = log_pyfunc(BlastRadiusModel(False), {"graph_snapshot": f"{D}/graph_snapshot.json"}, br_in, BR, "register_blast_radius_sim_path_only_v1", {"role": "challenger", "never_promote": "true"})
    c = mlflow.MlflowClient(registry_uri="databricks-uc")
    for n, ch, cl in [(FC, v_fc, v_lg), (BR, v_br, v_po)]:
        c.set_registered_model_alias(n, "champion", ch); c.set_registered_model_alias(n, "challenger", cl)

# Verify the @champion aliases reproduce the published tables
fc = spark.table(f"{S}.sim_capacity_forecast").toPandas().sort_values(["corridor_id", "month"]).reset_index(drop=True)
o = mlflow.pyfunc.load_model(f"models:/{FC}@champion").predict(fc[["corridor_id", "month"]])
cols = ["forecast_utilization_pct", "forecast_p10_pct", "forecast_p90_pct"]
print("forecast max abs err:", np.abs(o[cols].values.astype(float) - fc[cols].values.astype(float)).max(),
      "| exhaustion dates match:", all(str(a) == str(b) if b is not None and not pd.isna(b) else a is None for a, b in zip(o.exhaustion_date, fc.exhaustion_date)))
sfr = spark.table(f"{S}.sim_failure_result").toPandas().sort_values("sim_id").reset_index(drop=True)
ob = mlflow.pyfunc.load_model(f"models:/{BR}@champion").predict(sfr.rename(columns={"scenario_type": "resource_type"})[["resource_type", "resource_id"]])
print("blast radius services match:", (ob.affected_services.values == sfr.affected_services.values).all(),
      "| exposure match:", np.allclose(ob.sla_exposure_mrr, sfr.sla_exposure_mrr.astype(float), atol=0.005))